# 1) clone, install Archify

In [1]:
import subprocess

def node_major():
    try:
        out = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout
        return int(out.strip().lstrip("v").split(".")[0])
    except Exception:
        return 0

if node_major() < 18:
    print("Installing Node.js 22 ...")
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
    !apt-get install -y nodejs > /dev/null 2>&1

!node --version
![ -d /content/archify ] || git clone -q --depth 1 https://github.com/tt-a1i/archify.git /content/archify
!node /content/archify/archify/bin/archify.mjs doctor | tail -1
!mkdir -p /content/studio

v20.19.0
Archify is ready.


# 2) install cloudflared


In [2]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
!/content/cloudflared --version

cloudflared version 2026.9.1 (built 2026-09-11-13:35 UTC)


# 3) write studio server .py

In [3]:
%%writefile /content/studio/server.py
import http.cookies, json, os, re, secrets, subprocess, tempfile, threading, time, urllib.error, urllib.request, uuid
from http.server import ThreadingHTTPServer, BaseHTTPRequestHandler
from pathlib import Path
from urllib.parse import urlparse, unquote, parse_qs

HERE = Path(__file__).resolve().parent
CLI = os.environ.get("ARCHIFY_CLI", "/content/archify/archify/bin/archify.mjs")
ARCHIFY_DIR = Path(CLI).resolve().parents[1]          # .../archify/archify
DATA = Path(os.environ.get("STUDIO_DATA", HERE / "data"))
SITE = Path(os.environ.get("STUDIO_SITE", HERE / "site"))
PORT = int(os.environ.get("PORT", "8000"))
ACCESS_KEY = os.environ.get("STUDIO_KEY", "")            # empty = no access check

OPENAI_KEY = os.environ.get("OPENAI_API_KEY", "")
OPENAI_BASE = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1").rstrip("/")
OPENAI_STYLE = os.environ.get("OPENAI_API_STYLE", "responses")   # "responses" or "chat"
DEFAULT_MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.6-terra")
REASONING = os.environ.get("OPENAI_REASONING", "low")             # "" to omit
MAX_ROUNDS = int(os.environ.get("AI_MAX_ROUNDS", "4"))           # 1 draft + up to 3 repairs

TYPES = ["architecture", "sequence", "workflow", "dataflow", "lifecycle"]
QUALITIES = {"showcase", "standard"}
NAME_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}")
MAX_BODY = 1_000_000
render_lock = threading.Lock()
jobs, jobs_lock = {}, threading.Lock()

DATA.mkdir(parents=True, exist_ok=True)
SITE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("ARCHIFY_UPDATE_CHECK_DISABLED", "1")


# ------------------------------------------------------------------ Archify helpers
def valid_name(name):
    return bool(name and NAME_RE.fullmatch(name))


def slug(text, fallback="diagram"):
    s = re.sub(r"[^A-Za-z0-9]+", "-", text or "").strip("-").lower()[:48]
    return s or fallback


def unique_name(base):
    name, i = base, 2
    while (DATA / f"{name}.json").exists():
        name, i = f"{base}-{i}", i + 1
    return name


def list_diagrams():
    items = []
    for f in sorted(DATA.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True):
        try:
            src = json.loads(f.read_text())
            title, dtype = src.get("meta", {}).get("title", f.stem), src.get("diagram_type", "?")
        except Exception:
            title, dtype = f.stem, "?"
        items.append({"name": f.stem, "title": title, "type": dtype, "updated": int(f.stat().st_mtime),
                      "rendered": (SITE / f"{f.stem}.html").exists()})
    return items


def simplify(receipt):
    return [{"message": d.get("message", ""), "path": (d.get("subject") or {}).get("path"),
             "code": d.get("code"), "fixes": d.get("supportedFixes", [])}
            for d in receipt.get("diagnostics", [])]


def run_cli(args, timeout=90):
    r = subprocess.run(["node", CLI, *args], capture_output=True, text=True, timeout=timeout)
    try:
        return json.loads(r.stdout)
    except json.JSONDecodeError:
        return {"ok": False, "error": (r.stderr or r.stdout or "Archify gave no output.")[-2000:]}


def validate(dtype, obj, quality):
    with tempfile.TemporaryDirectory() as d:
        p = Path(d) / "candidate.json"
        p.write_text(json.dumps(obj, indent=2, ensure_ascii=False))
        receipt = run_cli(["validate", dtype, str(p), "--quality", quality, "--json"])
    diags = simplify(receipt)
    if not receipt.get("ok") and not diags:
        diags = [{"message": receipt.get("error", "Validation failed."), "path": None, "code": None, "fixes": []}]
    return bool(receipt.get("ok")), diags


def parse_source(source_text):
    try:
        src = json.loads(source_text)
    except json.JSONDecodeError as e:
        return None, {"ok": False, "stage": "json", "line": e.lineno, "column": e.colno,
                      "error": f"The JSON can't be read: {e.msg} (line {e.lineno}, column {e.colno})."}
    if not isinstance(src, dict):
        return None, {"ok": False, "stage": "json", "error": "The top level must be a JSON object { ... }."}
    if src.get("diagram_type") not in TYPES:
        return None, {"ok": False, "stage": "json", "error": f'"diagram_type" must be one of: {", ".join(TYPES)}.'}
    return src, None


def render(name, source_text, quality="showcase"):
    src, err = parse_source(source_text)
    if err:
        return err
    dtype = src["diagram_type"]
    (DATA / f"{name}.json").write_text(source_text)       # save even if rendering fails
    out = SITE / f"{name}.html"
    with render_lock:
        try:
            receipt = run_cli(["deliver", dtype, str(DATA / f"{name}.json"), str(out),
                               "--quality", quality, "--json"])
        except subprocess.TimeoutExpired:
            return {"ok": False, "stage": "render", "error": "Rendering took longer than 90 seconds and was stopped."}
    result = {"ok": bool(receipt.get("ok")), "type": dtype, "stage": receipt.get("stage"), "name": name}
    if result["ok"]:
        result["url"] = f"/view/{name}.html?v={int(time.time() * 1000)}"
    else:
        result["error"] = receipt.get("error", "Archify rejected this diagram.")
        result["diagnostics"] = simplify(receipt)
    return result


def guide(query):
    try:
        data = run_cli(["guide", query, "--json"], timeout=30)
    except Exception as e:
        return {"ok": False, "error": str(e)}
    rec = data.get("recommendation") or {}
    return {"ok": bool(rec), "confidence": data.get("confidence"), "type": rec.get("type"),
            "title": rec.get("title"), "summary": rec.get("summary"), "include": rec.get("include", [])}


# ------------------------------------------------------------------ GPT prompt
EXAMPLES = {"architecture": "web-app.architecture.json", "sequence": "cache-miss-request.sequence.json",
            "workflow": "agent-tool-call.workflow.json", "dataflow": "product-analytics.dataflow.json",
            "lifecycle": "agent-run.lifecycle.json"}

TYPE_NOTES = {
    "architecture": "- Components need explicit \"pos\": [x, y] and \"size\": [w, h]. Lay the main path left to right, "
                    "leave clear gaps (at least 70px) between boxes so labels fit, and put side services above or below it.\n"
                    "- Use \"boundaries\" only for real regions or trust zones.",
    "sequence": "- Participants are columns in call order. Messages need increasing \"y\" values (about 28-35px apart). "
                "Use variant \"return\" for responses. meta.viewBox height must be at least 480.\n"
                "- Set meta.column_fit to \"spread\" when participant labels are long or the viewBox is wide.",
    "workflow": "- New workflows use \"schema_version\": 2 and its readable layout contract; the example shows the fields.\n"
                "- \"mainPath\" lists the happy path in order; exceptions go in their own lane.",
    "dataflow": "- Nodes sit in \"stages\" (columns) and \"rows\". Flows go left to right; label them with the data that moves.",
    "lifecycle": "- Phase columns 0..4 occupy the main rail; an event/terminal column N in 0..2 aligns beneath main column N + 2.\n"
                 "- A recoverable state uses type \"failure\" plus a real transition back to the active state.",
}


def schema_version_for(dtype):
    return 2 if dtype == "workflow" else 1


def compact(obj):
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))


def system_prompt(dtype, quality):
    schema = json.loads((ARCHIFY_DIR / "schemas" / f"{dtype}.schema.json").read_text())
    common = json.loads((ARCHIFY_DIR / "schemas" / "common.schema.json").read_text())
    example = json.loads((ARCHIFY_DIR / "examples" / EXAMPLES[dtype]).read_text())
    example.get("meta", {}).pop("output", None)
    return f"""You write Archify diagram specifications. Archify compiles one typed JSON object into an interactive HTML diagram and validates it strictly.
Always answer with exactly one JSON object and nothing else: no markdown fences, no comments, no explanations.

Diagram type: {dtype}. Use "schema_version": {schema_version_for(dtype)} and "diagram_type": "{dtype}".

Authoring rules:
- Follow the JSON Schema below exactly. Never invent fields.
- Write fresh content from the user's request: new stable lowercase ids (letters, digits, dashes) and domain wording. The example only shows field shape; never copy its facts.
- One obvious main path, short side branches, at most 12 primary nodes, sparse but meaningful labels.
- Node/component types: frontend, backend, database, cloud, security, messagebus, external. Relationship variants: default, emphasis, security, dashed.
- Set meta.quality_profile to "{quality}". Omit meta.visual_preset, meta.subtitle, meta.legend and meta.engineering_profile unless the user asks for them. Add "animation": "trace" to meta only if the user wants motion.
- Start with automatic routes and label placement. Do not use via, channelX, channelY or labelAt until a diagnostic asks for one, and add at most one such control per repair.
- Keep exact product names, protocols, API paths and commands. Write labels in the user's language; set meta.locale only to "en" or "zh-CN", otherwise omit it.
- Optional "cards" hold supporting detail; prefer cards over extra edges.
{TYPE_NOTES[dtype]}

Repairs: when you receive validator diagnostics, change only the diagnosed subject, prefer one of its supportedFixes, keep everything else identical, and return the complete corrected JSON object.

JSON Schema for {dtype}:
{compact(schema)}

Shared definitions (common.schema.json):
{compact(common)}

Valid example ({dtype}, field shape only):
{compact(example)}"""


# ------------------------------------------------------------------ GPT calls
class AIError(Exception):
    pass


def _post(url, body):
    req = urllib.request.Request(url, data=json.dumps(body).encode(), method="POST", headers={
        "Authorization": f"Bearer {OPENAI_KEY}", "Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=300) as r:
            return json.load(r)
    except urllib.error.HTTPError as e:
        raw = e.read().decode(errors="ignore")
        try:
            msg = json.loads(raw).get("error", {}).get("message") or raw
        except Exception:
            msg = raw
        raise AIError(f"{e.code}: {msg[:600]}")
    except urllib.error.URLError as e:
        raise AIError(f"Couldn't reach {OPENAI_BASE}: {e.reason}")


rejected = {"reasoning": set(), "json": set()}   # models that refused an optional parameter


def call_llm(system, messages, model):
    """Returns (text, usage). Drops optional parameters a model rejects and remembers that."""
    json_mode = model not in rejected["json"]
    effort = REASONING if model not in rejected["reasoning"] else ""
    for _ in range(3):
        if OPENAI_STYLE == "chat":
            body = {"model": model, "messages": [{"role": "system", "content": system}] + messages}
            if json_mode:
                body["response_format"] = {"type": "json_object"}
            url = f"{OPENAI_BASE}/chat/completions"
        else:
            body = {"model": model, "instructions": system, "input": messages}
            if json_mode:
                body["text"] = {"format": {"type": "json_object"}}
            if effort:
                body["reasoning"] = {"effort": effort}
            url = f"{OPENAI_BASE}/responses"
        try:
            data = _post(url, body)
            break
        except AIError as e:
            text = str(e).lower()
            if e.args[0].startswith("400") and effort and "reasoning" in text:
                effort = ""
                rejected["reasoning"].add(model)
                continue
            if e.args[0].startswith("400") and json_mode and ("format" in text or "json" in text):
                json_mode = False
                rejected["json"].add(model)
                continue
            raise
    if OPENAI_STYLE == "chat":
        text = (data.get("choices") or [{}])[0].get("message", {}).get("content") or ""
    else:
        text = data.get("output_text") or "".join(
            c.get("text", "") for item in data.get("output", []) if item.get("type") == "message"
            for c in item.get("content", []) if c.get("type") in ("output_text", "text"))
        if not text and data.get("status") == "incomplete":
            raise AIError(f"The model stopped early: {data.get('incomplete_details')}")
    usage = data.get("usage") or {}
    tokens = (usage.get("input_tokens") or usage.get("prompt_tokens") or 0,
              usage.get("output_tokens") or usage.get("completion_tokens") or 0)
    return text, tokens


def extract_json(text):
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t)
    start, end = t.find("{"), t.rfind("}")
    if start < 0 or end <= start:
        raise ValueError("no JSON object found in the reply")
    return json.loads(t[start:end + 1])


def describe_problems(diags):
    lines = []
    for i, d in enumerate(diags[:12], 1):
        fixes = "; ".join(d["fixes"]) if d["fixes"] else "none listed"
        lines.append(f"{i}. [{d.get('code') or 'error'}] {d['message']} (at {d.get('path') or 'n/a'}). supportedFixes: {fixes}")
    return "\n".join(lines)


def ai_job(job, req):
    def log(msg, **extra):
        with jobs_lock:
            job["log"].append({"t": round(time.time() - job["started"], 1), "msg": msg, **extra})

    try:
        mode = req.get("mode", "create")                       # create | edit | fix
        quality = req.get("quality") if req.get("quality") in QUALITIES else "showcase"
        model = (req.get("model") or DEFAULT_MODEL).strip()
        description = (req.get("description") or "").strip()
        current = None
        if mode in ("edit", "fix"):
            current, err = parse_source(req.get("source") or "")
            if err:
                raise AIError("The editor JSON has to be valid before GPT can change it. " + err["error"])
            dtype = current["diagram_type"]
        else:
            dtype = req.get("type") or "auto"
            if dtype not in TYPES:
                g = guide(description)
                dtype = g.get("type") if g.get("type") in TYPES else "architecture"
                log(f"Picked a {dtype} diagram for this description.")

        system = system_prompt(dtype, quality)
        if mode == "create":
            first = f"Create an Archify {dtype} diagram as a JSON object for this request:\n\n{description}"
        elif mode == "edit":
            first = (f"Here is the current Archify {dtype} JSON:\n{json.dumps(current, ensure_ascii=False)}\n\n"
                     f"Apply this change and return the complete updated JSON object. Keep ids and the layout of "
                     f"unrelated parts stable.\n\nChange: {description}")
        else:
            ok, diags = validate(dtype, current, quality)
            if ok:
                job.update(status="done", result={"ok": True, "unchanged": True, "type": dtype,
                                                  "source": json.dumps(current, indent=2, ensure_ascii=False)})
                log("The current JSON already passes every check. Nothing to fix.")
                return
            first = (f"Here is an Archify {dtype} JSON that fails validation:\n{json.dumps(current, ensure_ascii=False)}\n\n"
                     f"Validator diagnostics:\n{describe_problems(diags)}\n\nReturn the complete corrected JSON object.")

        best, best_count, candidate, stale = None, None, None, 0
        feedback = None
        for rnd in range(1, MAX_ROUNDS + 1):
            messages = [{"role": "user", "content": first}]
            if feedback:
                messages += [{"role": "assistant", "content": json.dumps(candidate, ensure_ascii=False)},
                             {"role": "user", "content": feedback}]
            log(f"Round {rnd}: asking {model}…")
            text, (tin, tout) = call_llm(system, messages, model)
            job["tokens"][0] += tin
            job["tokens"][1] += tout
            try:
                obj = extract_json(text)
            except ValueError as e:
                log(f"Round {rnd}: the reply wasn't valid JSON ({e}).", problems=None)
                feedback = f"Your reply was not a valid JSON object ({e}). Return only the complete JSON object."
                candidate = candidate or {}
                stale += 1
                if stale >= 2:
                    break
                continue
            obj["diagram_type"] = dtype
            ok, diags = validate(dtype, obj, quality)
            candidate = obj
            log(f"Round {rnd}: {'passed all checks' if ok else f'{len(diags)} problem(s)'}.", problems=len(diags))
            if best_count is None or len(diags) < best_count:
                best, best_count, stale = obj, len(diags), 0
            else:
                stale += 1
            if ok or stale >= 2:
                break
            feedback = ("Archify validation failed with these diagnostics:\n" + describe_problems(diags) +
                        "\n\nFix only these subjects and return the complete corrected JSON object.")

        if best is None:
            raise AIError("GPT never returned a usable JSON object. Try again or pick another model.")
        if mode == "create":
            name = unique_name(slug(best.get("meta", {}).get("title", ""), dtype))
        else:
            name = req.get("name") if valid_name(req.get("name")) else unique_name(dtype)
        source = json.dumps(best, indent=2, ensure_ascii=False)
        result = render(name, source, quality)
        result.update(source=source, rounds=len([l for l in job["log"] if "problems" in l]))
        job.update(status="done", result=result)
        log("Saved and rendered." if result["ok"] else
            f"Saved the best attempt; {len(result.get('diagnostics') or [])} problem(s) remain.")
    except Exception as e:
        job.update(status="error", error=str(e))
        log(f"Stopped: {e}")


def start_job(req):
    job = {"id": uuid.uuid4().hex[:12], "status": "running", "log": [], "tokens": [0, 0],
           "started": time.time(), "result": None, "error": None}
    with jobs_lock:
        jobs[job["id"]] = job
        for old in sorted(jobs.values(), key=lambda j: j["started"])[:-30]:   # keep the last 30
            jobs.pop(old["id"], None)
    threading.Thread(target=ai_job, args=(job, req), daemon=True).start()
    return job


# ------------------------------------------------------------------ HTTP
LOCKED_PAGE = """<!doctype html><meta charset="utf-8"><title>Archify Studio</title>
<body style="font:16px system-ui;max-width:560px;margin:80px auto;padding:0 20px;color:#dce5ef;background:#0e141c;color-scheme:dark">
<h1>Open the link from Colab</h1><p>This studio is private. Use the full link printed by the tunnel cell in your
Colab notebook; it includes an access key.</p></body>"""


class Handler(BaseHTTPRequestHandler):
    server_version = "ArchifyStudio/2.0"

    def log_message(self, fmt, *args):
        path = self.path.split("?")[0]
        if not path.startswith("/api/ai/jobs/"):
            print(f"[{time.strftime('%H:%M:%S')}] {self.command} {path} -> {args[1] if len(args) > 1 else ''}", flush=True)

    def send(self, status, body, ctype="application/json; charset=utf-8", extra=None):
        if isinstance(body, (dict, list)):
            body = json.dumps(body, ensure_ascii=False)
        if isinstance(body, str):
            body = body.encode()
        self.send_response(status)
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.send_header("X-Content-Type-Options", "nosniff")
        for k, v in (extra or {}).items():
            self.send_header(k, v)
        self.end_headers()
        if self.command != "HEAD":
            self.wfile.write(body)

    def authorized(self):
        if not ACCESS_KEY:
            return True
        c = http.cookies.SimpleCookie(self.headers.get("Cookie", ""))
        return "studio_key" in c and secrets.compare_digest(c["studio_key"].value, ACCESS_KEY)

    def gate(self):
        """Returns True if the request may continue. Handles ?key= login."""
        url = urlparse(self.path)
        key = (parse_qs(url.query).get("key") or [""])[0]
        if ACCESS_KEY and key and secrets.compare_digest(key, ACCESS_KEY):
            self.send(302, "", "text/plain", {
                "Location": url.path or "/",
                "Set-Cookie": f"studio_key={ACCESS_KEY}; Path=/; HttpOnly; SameSite=Lax; Max-Age=604800"})
            return False
        if self.authorized():
            return True
        if url.path.startswith("/api/"):
            self.send(401, {"ok": False, "error": "Open the studio with the link printed in Colab."})
        else:
            self.send(401, LOCKED_PAGE, "text/html; charset=utf-8")
        return False

    def read_json(self):
        length = int(self.headers.get("Content-Length") or 0)
        if length > MAX_BODY:
            raise ValueError("Request is larger than 1 MB.")
        return json.loads(self.rfile.read(length) or b"{}")

    def do_HEAD(self):
        self.do_GET()

    def do_GET(self):
        if not self.gate():
            return
        path = unquote(urlparse(self.path).path)
        if path in ("/", "/index.html"):
            return self.send(200, (HERE / "index.html").read_bytes(), "text/html; charset=utf-8")
        if path == "/api/starters":
            return self.send(200, (HERE / "starters.json").read_bytes())
        if path == "/api/diagrams":
            return self.send(200, list_diagrams())
        if path == "/api/ai/status":
            return self.send(200, {"enabled": bool(OPENAI_KEY), "model": DEFAULT_MODEL,
                                   "base": urlparse(OPENAI_BASE).netloc, "maxRounds": MAX_ROUNDS})

        m = re.fullmatch(r"/api/ai/jobs/([a-f0-9]{12})", path)
        if m:
            with jobs_lock:
                job = jobs.get(m[1])
                snap = None if not job else {k: job[k] for k in ("id", "status", "log", "tokens", "result", "error")}
            return self.send(200, snap) if snap else self.send(404, {"error": "Unknown job."})

        m = re.fullmatch(r"/api/diagrams/([^/]+)", path)
        if m and valid_name(m[1]):
            f = DATA / f"{m[1]}.json"
            return self.send(200, {"name": m[1], "source": f.read_text()}) if f.exists() \
                else self.send(404, {"error": "No saved diagram with that name."})

        m = re.fullmatch(r"/(view|download)/([^/]+)\.(html|json)", path)
        if m and valid_name(m[2]):
            kind, name, ext = m.groups()
            f = (SITE if ext == "html" else DATA) / f"{name}.{ext}"
            if not f.exists():
                return self.send(404, "Not rendered yet.", "text/plain; charset=utf-8")
            ctype = "text/html; charset=utf-8" if ext == "html" else "application/json; charset=utf-8"
            extra = {"Content-Disposition": f'attachment; filename="{name}.{ext}"'} if kind == "download" else None
            return self.send(200, f.read_bytes(), ctype, extra)

        self.send(404, {"error": "Not found"})

    def do_POST(self):
        if not self.gate():
            return
        path = urlparse(self.path).path
        try:
            body = self.read_json()
        except Exception as e:
            return self.send(400, {"ok": False, "error": f"Bad request: {e}"})
        if path == "/api/render":
            name = (body.get("name") or "").strip()
            if not valid_name(name):
                return self.send(400, {"ok": False, "stage": "name",
                                       "error": "Use a name with letters, numbers, - or _ (up to 64 characters)."})
            quality = body.get("quality") if body.get("quality") in QUALITIES else "showcase"
            return self.send(200, render(name, body.get("source") or "", quality))
        if path == "/api/guide":
            q = (body.get("q") or "").strip()[:500]
            return self.send(200, guide(q) if q else {"ok": False, "error": "Describe what you want to show."})
        if path == "/api/ai/jobs":
            if not OPENAI_KEY:
                return self.send(400, {"ok": False, "error": "No OpenAI API key. Add OPENAI_API_KEY in Colab and rerun the server cell."})
            if body.get("mode", "create") in ("create", "edit") and not (body.get("description") or "").strip():
                return self.send(400, {"ok": False, "error": "Describe the diagram or the change first."})
            job = start_job(body)
            return self.send(200, {"ok": True, "id": job["id"]})
        self.send(404, {"error": "Not found"})

    def do_DELETE(self):
        if not self.gate():
            return
        m = re.fullmatch(r"/api/diagrams/([^/]+)", urlparse(self.path).path)
        if not (m and valid_name(m[1])):
            return self.send(404, {"error": "Not found"})
        for f in (DATA / f"{m[1]}.json", SITE / f"{m[1]}.html"):
            f.unlink(missing_ok=True)
        self.send(200, {"ok": True})


if __name__ == "__main__":
    print(f"Archify Studio on http://127.0.0.1:{PORT}  (data: {DATA}, GPT: {'on' if OPENAI_KEY else 'off'})", flush=True)
    ThreadingHTTPServer(("127.0.0.1", PORT), Handler).serve_forever()

Overwriting /content/studio/server.py


# 4) write studio index html page

In [4]:
%%writefile /content/studio/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Archify Studio</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500&family=IBM+Plex+Sans:wght@400;500;600&display=swap" rel="stylesheet">
<style>
  :root, :root[data-theme="dark"] {
    color-scheme: dark;
    --bg: #0e141c; --panel: #151e29; --code-bg: #111923; --ink: #dce5ef; --muted: #8b9bb0; --line: #273648;
    --accent: #5b86ff; --accent-hover: #4a74ec; --accent-ink: #ffffff;
    --bad: #ff7a8a; --bad-bg: #2e161d; --good: #56c995; --good-bg: #11271c;
    --head-bg: #0a0f15; --head-ink: #ffffff; --head-muted: #8fa1b6;
    --ctl-bg: #1c2836; --ctl-line: #31445a; --ctl-hover: #253547;
    --field-bg: #0f1720; --active: #1f3155; --ai-bg: #131d31;
    --stale-bg: #33290f; --stale-ink: #f1d68d; --stale-line: #574616; --chip-bg: #0b1017;
  }
  :root[data-theme="light"] {
    color-scheme: light;
    --bg: #e8ecf1; --panel: #ffffff; --code-bg: #f6f8fb; --ink: #1b2a3a; --muted: #5b6b7c; --line: #cfd7e0;
    --accent: #2f5bea; --accent-hover: #2549c4; --accent-ink: #ffffff;
    --bad: #b42335; --bad-bg: #fbeef0; --good: #2e7d5b; --good-bg: #eaf5ef;
    --head-bg: #1b2a3a; --head-ink: #ffffff; --head-muted: #9fb0c3;
    --ctl-bg: #26394d; --ctl-line: #3b526a; --ctl-hover: #31485f;
    --field-bg: #ffffff; --active: #e3eafc; --ai-bg: #f0f4ff;
    --stale-bg: #fff6dd; --stale-ink: #6b5200; --stale-line: #ecdca8; --chip-bg: #ffffff;
  }
  :root {
    --sans: "IBM Plex Sans", system-ui, -apple-system, "Segoe UI", sans-serif;
    --mono: "IBM Plex Mono", ui-monospace, "SFMono-Regular", Menlo, Consolas, monospace;
  }
  * { box-sizing: border-box; }
  html, body { height: 100%; margin: 0; }
  body { background: var(--bg); color: var(--ink); font: 14px/1.45 var(--sans); display: flex; flex-direction: column; }
  button, input, select, textarea { font: inherit; color: inherit; }
  ::placeholder { color: var(--muted); opacity: 1; }
  :focus-visible { outline: 2px solid var(--accent); outline-offset: 2px; }
  a { color: var(--accent); }

  header { display: flex; flex-wrap: wrap; align-items: center; gap: 10px; padding: 10px 16px;
           background: var(--head-bg); color: var(--head-ink); border-bottom: 1px solid var(--line); }
  .brand { font-weight: 600; font-size: 16px; margin-right: 8px; }
  .brand small { font-weight: 400; color: var(--head-muted); margin-left: 6px; }
  header label { color: var(--head-muted); font-size: 13px; }
  header input, header select { background: var(--ctl-bg); border: 1px solid var(--ctl-line); border-radius: 6px;
           padding: 6px 9px; color: var(--head-ink); }
  header input { width: 180px; font-family: var(--mono); font-size: 13px; }
  .spacer { flex: 1; }
  .btn { border: 1px solid transparent; border-radius: 6px; padding: 6px 12px; cursor: pointer;
         background: var(--ctl-bg); color: var(--head-ink); text-decoration: none; display: inline-flex; align-items: center; gap: 6px; }
  .btn:hover { background: var(--ctl-hover); }
  .btn.primary { background: var(--accent); color: var(--accent-ink); font-weight: 600; }
  .btn.primary:hover { background: var(--accent-hover); }
  .btn[aria-disabled="true"], .btn:disabled { opacity: .45; pointer-events: none; }
  .btn.light { background: var(--panel); color: var(--ink); border-color: var(--line); }
  .btn.light:hover { background: var(--code-bg); }
  kbd { font-family: var(--mono); font-size: 11px; opacity: .75; }

  main { flex: 1; min-height: 0; display: grid; gap: 12px; padding: 12px;
         grid-template-columns: 250px minmax(340px, 1fr) minmax(380px, 1.35fr); }
  section { background: var(--panel); border: 1px solid var(--line); border-radius: 8px;
            min-height: 0; display: flex; flex-direction: column; overflow: hidden; }
  h2 { font-size: 13px; font-weight: 600; margin: 0; padding: 10px 12px; border-bottom: 1px solid var(--line);
       display: flex; align-items: center; gap: 8px; }
  h2 .grow { flex: 1; }
  #previewName { font-family: var(--mono); font-weight: 400; color: var(--muted); }

  /* Sidebar */
  #list { list-style: none; margin: 0; padding: 6px; overflow: auto; flex: 1; }
  #list li { display: flex; align-items: center; border-radius: 6px; }
  #list li.active { background: var(--active); }
  #list button.open { flex: 1; text-align: left; background: none; border: 0; padding: 7px 8px; cursor: pointer; min-width: 0; }
  #list .t { display: block; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
  #list .n { display: block; font-family: var(--mono); font-size: 11.5px; color: var(--muted); }
  #list button.del { background: none; border: 0; color: var(--muted); cursor: pointer; padding: 6px 8px; border-radius: 4px; }
  #list button.del:hover { color: var(--bad); background: var(--bad-bg); }
  .empty { color: var(--muted); padding: 10px; }
  .guide { border-top: 1px solid var(--line); padding: 12px; display: grid; gap: 8px; }
  .guide p { margin: 0; }
  .guide textarea { width: 100%; min-height: 64px; resize: vertical; border: 1px solid var(--line);
                    border-radius: 6px; padding: 7px; background: var(--field-bg); }
  #guideOut { font-size: 13px; }
  #guideOut ul { margin: 4px 0 6px; padding-left: 18px; }
  #guideOut .fix { color: var(--muted); }

  /* GPT panel */
  .ai { border-bottom: 1px solid var(--line); background: var(--ai-bg); }
  .ai summary { cursor: pointer; padding: 9px 12px; font-weight: 600; list-style-position: inside; }
  .ai summary small { font-weight: 400; color: var(--muted); margin-left: 6px; }
  .ai-body { padding: 0 12px 12px; display: grid; gap: 8px; }
  .ai-body textarea { width: 100%; min-height: 70px; resize: vertical; border: 1px solid var(--line);
                      border-radius: 6px; padding: 8px; background: var(--field-bg); }
  .ai-row { display: flex; flex-wrap: wrap; gap: 6px; align-items: center; }
  .ai-row label { font-size: 12.5px; color: var(--muted); }
  .ai-row select, .ai-row input { border: 1px solid var(--line); border-radius: 6px; padding: 5px 8px; background: var(--field-bg); }
  .ai-row input { width: 150px; font-family: var(--mono); font-size: 12.5px; }
  .ai-off { font-size: 13px; color: var(--muted); }

  /* Editor */
  .toolbar { display: flex; gap: 6px; padding: 8px 10px; border-bottom: 1px solid var(--line); flex-wrap: wrap; align-items: center; }
  .toolbar .pos { margin-left: auto; font-family: var(--mono); font-size: 12px; color: var(--muted); }
  #editor { flex: 1; min-height: 240px; border: 0; resize: none; padding: 12px; background: var(--code-bg);
            font: 12.5px/1.55 var(--mono); tab-size: 2; white-space: pre; overflow: auto; }
  #editor:focus-visible { outline: none; box-shadow: inset 0 0 0 2px var(--accent); }
  #status { border-top: 1px solid var(--line); padding: 10px 12px; max-height: 38%; overflow: auto; font-size: 13px; }
  #status.ok { background: var(--good-bg); color: var(--good); }
  #status.bad { background: var(--bad-bg); color: var(--bad); }
  #status .diag { color: var(--ink); margin: 8px 0 0; padding-left: 10px; border-left: 3px solid var(--bad); }
  #status code { font-family: var(--mono); font-size: 12px; background: var(--chip-bg); padding: 1px 4px; border-radius: 3px; }
  #status .fix { color: var(--muted); }
  #status .log { color: var(--ink); font-family: var(--mono); font-size: 12px; margin: 6px 0 0; white-space: pre-wrap; }
  #status .actions { margin-top: 8px; }

  /* Preview */
  .preview-wrap { position: relative; flex: 1; min-height: 0; background: var(--panel); }
  #preview { width: 100%; height: 100%; border: 0; display: block; background: transparent; }
  #placeholder { position: absolute; inset: 0; display: grid; place-content: center; text-align: center;
                 color: var(--muted); padding: 24px; }
  #placeholder strong { color: var(--ink); display: block; font-size: 15px; margin-bottom: 4px; }
  #stale { display: none; padding: 6px 12px; font-size: 12.5px; background: var(--stale-bg); color: var(--stale-ink);
           border-bottom: 1px solid var(--stale-line); }

  @media (max-width: 1100px) {
    html, body { height: auto; }
    main { display: flex; flex-direction: column; }
    section { flex: none; }
    #list { flex: none; max-height: 220px; }
    section.edit { height: 70vh; }
    section.view { height: 75vh; }
    header input { width: 160px; }
  }
  @media (prefers-reduced-motion: no-preference) {
    .btn { transition: background-color .12s; }
  }
</style>
<script>
  try { document.documentElement.dataset.theme = localStorage.getItem("studio-theme") || "dark"; }
  catch (e) { document.documentElement.dataset.theme = "dark"; }
</script>
</head>
<body>
<header>
  <div class="brand">Archify Studio<small>on Colab</small></div>
  <label for="name">Name</label>
  <input id="name" spellcheck="false" autocomplete="off" aria-describedby="nameHelp">
  <span id="nameHelp" hidden>Letters, numbers, dashes and underscores</span>
  <label for="newType">New</label>
  <select id="newType" aria-label="Start a new diagram">
    <option value="">Choose a type…</option>
    <option value="architecture">Architecture</option>
    <option value="sequence">Sequence</option>
    <option value="workflow">Workflow</option>
    <option value="dataflow">Data flow</option>
    <option value="lifecycle">Lifecycle</option>
  </select>
  <label for="quality">Checks</label>
  <select id="quality" title="Strict runs every Archify layout check; Basic is more forgiving">
    <option value="showcase">Strict</option>
    <option value="standard">Basic</option>
  </select>
  <button class="btn primary" id="renderBtn" title="Ctrl/Cmd + Enter">Render <kbd>⌘↵</kbd></button>
  <div class="spacer"></div>
  <button class="btn" id="themeBtn" type="button"></button>
  <a class="btn" id="openLink" target="_blank" rel="noopener" aria-disabled="true">Open page</a>
  <label>Download</label>
  <a class="btn" id="dlHtml" aria-disabled="true" aria-label="Download HTML">HTML</a>
  <a class="btn" id="dlJson" aria-disabled="true" aria-label="Download JSON">JSON</a>
</header>

<main>
  <section class="side" aria-label="Your diagrams">
    <h2><span class="grow">Your diagrams</span></h2>
    <ul id="list"></ul>
    <div class="guide">
      <p><strong>Not sure which type?</strong></p>
      <textarea id="guideQ" placeholder="Describe what you want to show, e.g. how a login request moves through our services"></textarea>
      <button class="btn light" id="guideBtn">Suggest a type</button>
      <div id="guideOut" aria-live="polite"></div>
    </div>
  </section>

  <section class="edit" aria-label="Diagram source">
    <details class="ai" id="aiPanel" open>
      <summary>Create with GPT<small id="aiInfo"></small></summary>
      <div class="ai-body" id="aiBody">
        <textarea id="aiText" placeholder="Describe the system, e.g. Browser → API → Redis cache → PostgreSQL fallback, with JWT auth and an async email worker"></textarea>
        <div class="ai-row">
          <label for="aiType">Type</label>
          <select id="aiType">
            <option value="auto">Auto</option>
            <option value="architecture">Architecture</option>
            <option value="sequence">Sequence</option>
            <option value="workflow">Workflow</option>
            <option value="dataflow">Data flow</option>
            <option value="lifecycle">Lifecycle</option>
          </select>
          <label for="aiModel">Model</label>
          <input id="aiModel" list="models" spellcheck="false">
          <datalist id="models">
            <option value="gpt-5.6-terra"><option value="gpt-5.6-luna"><option value="gpt-5.6-sol"><option value="gpt-6-astra">
          </datalist>
        </div>
        <div class="ai-row">
          <button class="btn primary" id="aiCreate">Create diagram</button>
          <button class="btn light" id="aiEdit" title="Uses the text above as an instruction, e.g. add Redis, move auth to the left">Change current diagram</button>
        </div>
      </div>
    </details>
    <div class="toolbar">
      <button class="btn light" id="formatBtn">Format JSON</button>
      <button class="btn light" id="uploadBtn">Upload .json</button>
      <input type="file" id="file" accept=".json,application/json" hidden>
      <a class="btn light" href="https://github.com/tt-a1i/archify/blob/main/archify/schemas/README.md" target="_blank" rel="noopener">Field reference</a>
      <span class="pos" id="pos">1:1</span>
    </div>
    <textarea id="editor" spellcheck="false" autocapitalize="off" autocomplete="off" aria-label="Diagram JSON"></textarea>
    <div id="status" role="status" aria-live="polite">Edit the JSON, then press Render.</div>
  </section>

  <section class="view" aria-label="Preview">
    <h2><span class="grow">Preview</span><span id="previewName"></span></h2>
    <div id="stale">This is the last version that rendered. Fix the problems on the left to update it.</div>
    <div class="preview-wrap">
      <iframe id="preview" title="Rendered diagram"></iframe>
      <div id="placeholder"><div><strong>Nothing rendered yet</strong>Press Render to draw this diagram.</div></div>
    </div>
  </section>
</main>

<script>
const $ = (id) => document.getElementById(id);
const editor = $("editor"), nameInput = $("name"), statusBox = $("status");
let starters = {}, current = null, dirty = false, diagrams = [];

// ---------- formatting: objects expanded, short items kept on one line ----------
function compact(v) {
  if (v === null || typeof v !== "object") return JSON.stringify(v);
  if (Array.isArray(v)) return "[" + v.map(compact).join(", ") + "]";
  return "{" + Object.keys(v).map(k => JSON.stringify(k) + ": " + compact(v[k])).join(", ") + "}";
}
function pretty(v, ind = "") {
  if (v === null || typeof v !== "object") return JSON.stringify(v);
  const c = compact(v);
  if (ind.length >= 4 && c.length <= 150) return c;
  if (ind.length >= 2 && c.length <= 90) return c;
  const next = ind + "  ";
  if (Array.isArray(v)) {
    if (!v.length) return "[]";
    return "[\n" + v.map(x => next + pretty(x, next)).join(",\n") + "\n" + ind + "]";
  }
  const keys = Object.keys(v);
  if (!keys.length) return "{}";
  return "{\n" + keys.map(k => next + JSON.stringify(k) + ": " + pretty(v[k], next)).join(",\n") + "\n" + ind + "}";
}

// ---------- helpers ----------
const cleanName = (s) => (s || "").trim().replace(/\.json$/i, "").replace(/[^A-Za-z0-9_-]+/g, "-")
                                  .replace(/^[-_]+/, "").slice(0, 64) || "my-diagram";
function uniqueName(base) {
  const taken = new Set(diagrams.map(d => d.name));
  if (!taken.has(base)) return base;
  let i = 2; while (taken.has(`${base}-${i}`)) i++;
  return `${base}-${i}`;
}
function esc(s) { return String(s ?? "").replace(/[&<>"']/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c])); }
function setStatus(kind, html) { statusBox.className = kind || ""; statusBox.innerHTML = html; }
function confirmDiscard() { return !dirty || confirm("You have changes that haven't been rendered. Discard them?"); }
async function api(path, opts = {}) {
  const res = await fetch(path, { headers: { "Content-Type": "application/json" }, ...opts });
  if (res.status === 401) { location.reload(); throw new Error("Not signed in"); }
  return res.json();
}
let aiEnabled = false, busy = false;
function setBusy(on) {
  busy = on;
  ["renderBtn", "aiCreate", "aiEdit"].forEach(id => { const el = $(id); if (el) el.setAttribute("aria-disabled", on ? "true" : "false"); });
  document.querySelectorAll("#status button").forEach(b => b.disabled = on);
}

// ---------- theme ----------
let theme = document.documentElement.dataset.theme === "light" ? "light" : "dark";
function withTheme(url) {
  const u = new URL(url, location.origin);
  u.searchParams.set("theme", theme);
  return u.pathname + u.search;
}
function applyTheme() {
  document.documentElement.dataset.theme = theme;
  try { localStorage.setItem("studio-theme", theme); } catch (e) {}
  const btn = $("themeBtn");
  btn.textContent = theme === "dark" ? "Light mode" : "Dark mode";
  btn.title = `Switch the studio and preview to ${theme === "dark" ? "light" : "dark"} mode`;
  const src = $("preview").getAttribute("src");
  if (src) $("preview").src = withTheme(src);
  const open = $("openLink");
  if (open.getAttribute("aria-disabled") === "false") open.href = withTheme(open.getAttribute("href"));
}
$("themeBtn").addEventListener("click", () => { theme = theme === "dark" ? "light" : "dark"; applyTheme(); });
applyTheme();

function setLinks(name, rendered) {
  const set = (el, href, on) => { el.href = on ? href : "#"; el.setAttribute("aria-disabled", on ? "false" : "true"); };
  set($("openLink"), withTheme(`/view/${name}.html`), rendered);
  set($("dlHtml"), `/download/${name}.html`, rendered);
  set($("dlJson"), `/download/${name}.json`, !!diagrams.find(d => d.name === name));
}
function showPreview(url, name) {
  $("preview").src = withTheme(url);
  $("placeholder").style.display = "none";
  $("previewName").textContent = name;
}
function clearPreview() {
  $("preview").removeAttribute("src");
  $("placeholder").style.display = "grid";
  $("previewName").textContent = "";
  $("stale").style.display = "none";
}

// ---------- list ----------
async function refreshList() {
  diagrams = await api("/api/diagrams");
  const list = $("list");
  if (!diagrams.length) { list.innerHTML = `<li class="empty">No diagrams yet. Render one to save it here.</li>`; return; }
  list.innerHTML = diagrams.map(d => `
    <li class="${d.name === current ? "active" : ""}">
      <button class="open" data-name="${esc(d.name)}" title="Open ${esc(d.name)}">
        <span class="t">${esc(d.title)}</span><span class="n">${esc(d.name)} · ${esc(d.type)}</span>
      </button>
      <button class="del" data-name="${esc(d.name)}" aria-label="Delete ${esc(d.name)}">✕</button>
    </li>`).join("");
}
$("list").addEventListener("click", async (e) => {
  const btn = e.target.closest("button"); if (!btn) return;
  const name = btn.dataset.name;
  if (btn.classList.contains("del")) {
    if (!confirm(`Delete "${name}"? This removes its JSON and rendered HTML.`)) return;
    await api(`/api/diagrams/${name}`, { method: "DELETE" });
    if (name === current) { current = null; clearPreview(); }
    await refreshList(); setLinks(nameInput.value, false);
    return;
  }
  if (!confirmDiscard()) return;
  await openDiagram(name);
});

async function openDiagram(name) {
  const data = await api(`/api/diagrams/${name}`);
  if (data.error) { setStatus("bad", esc(data.error)); return; }
  editor.value = data.source; nameInput.value = name; current = name; dirty = false;
  const meta = diagrams.find(d => d.name === name);
  $("stale").style.display = "none";
  if (meta && meta.rendered) { showPreview(`/view/${name}.html?v=${Date.now()}`, name); setStatus("", "Opened. Edit and press Render to update."); }
  else { clearPreview(); setStatus("", "This diagram hasn't rendered successfully yet. Press Render to check it."); }
  setLinks(name, meta && meta.rendered);
  refreshList();
}

function newFrom(type) {
  if (!starters[type] || !confirmDiscard()) return;
  const name = uniqueName(`my-${type}`);
  editor.value = pretty(starters[type]) + "\n";
  nameInput.value = name; current = name; dirty = true;
  clearPreview(); setLinks(name, false); refreshList();
  setStatus("", `Started a new ${type} diagram from a template. Change the labels and connections, then press Render.`);
  editor.focus();
}
$("newType").addEventListener("change", (e) => { newFrom(e.target.value); e.target.value = ""; });

// ---------- render ----------
function selectLine(line, col) {
  const lines = editor.value.split("\n");
  let start = 0; for (let i = 0; i < line - 1 && i < lines.length; i++) start += lines[i].length + 1;
  const end = start + (lines[line - 1] || "").length;
  editor.focus(); editor.setSelectionRange(start, end);
  editor.scrollTop = Math.max(0, (line - 5) * 19);
}

async function render() {
  const name = cleanName(nameInput.value);
  nameInput.value = name;
  if (busy) return;
  setBusy(true);
  setStatus("", "Rendering…");
  let r;
  try {
    r = await api("/api/render", { method: "POST", body: JSON.stringify({ name, source: editor.value, quality: $("quality").value }) });
  } catch (e) {
    setStatus("bad", `Couldn't reach the Colab server. Check that the server and tunnel cells are still running in Colab. (${esc(e.message)})`);
    setBusy(false);
    return;
  }
  setBusy(false);
  showRenderResult(r, name);
  await refreshList();
  const meta = diagrams.find(d => d.name === name);
  setLinks(name, meta && meta.rendered);
}

function showRenderResult(r, name, prefix = "") {
  current = name;
  if (r.ok) {
    dirty = false;
    showPreview(r.url, name);
    $("stale").style.display = "none";
    setStatus("ok", `${prefix}Rendered ${esc(r.type)} diagram “${esc(name)}” at ${new Date().toLocaleTimeString()}. Saved.`);
  } else {
    const diags = r.diagnostics || [];
    let html = prefix + (diags.length
      ? `<strong>Archify found ${diags.length} problem${diags.length > 1 ? "s" : ""}.</strong> Your JSON was saved; the preview updates once these are fixed.`
      : `<strong>Can't render yet.</strong> ${esc(r.error || "")}`);
    for (const d of diags) {
      html += `<div class="diag">${esc(d.message)}${d.path ? ` <code>${esc(d.path)}</code>` : ""}` +
              (d.fixes || []).map(f => `<div class="fix">Fix: ${esc(f)}</div>`).join("") + `</div>`;
    }
    if (aiEnabled && r.stage !== "json" && r.stage !== "name") {
      html += `<div class="actions"><button class="btn light" id="fixBtn">Fix with GPT</button></div>`;
    }
    setStatus("bad", html);
    const fix = $("fixBtn"); if (fix) fix.addEventListener("click", () => runAI("fix"));
    if (r.line) selectLine(r.line, r.column);
    const hasPreview = $("preview").getAttribute("src") && $("previewName").textContent === name;
    $("stale").style.display = hasPreview ? "block" : "none";
  }
}

// ---------- GPT ----------
async function runAI(mode) {
  if (busy) return;
  const description = $("aiText").value.trim();
  if (mode !== "fix" && !description) { setStatus("bad", "Describe the diagram (or the change) in the GPT box first."); $("aiText").focus(); return; }
  if (mode === "create" && !confirmDiscard()) return;
  const payload = { mode, description, type: $("aiType").value, model: $("aiModel").value.trim(),
                    quality: $("quality").value, name: cleanName(nameInput.value), source: editor.value };
  setBusy(true);
  const label = { create: "Creating a diagram", edit: "Changing the diagram", fix: "Fixing the problems" }[mode];
  setStatus("", `<strong>${label} with GPT…</strong><div class="log">Starting…</div>`);
  let start;
  try {
    start = await api("/api/ai/jobs", { method: "POST", body: JSON.stringify(payload) });
  } catch (e) { setBusy(false); setStatus("bad", `Couldn't reach the Colab server. (${esc(e.message)})`); return; }
  if (!start.ok) { setBusy(false); setStatus("bad", esc(start.error)); return; }

  const t0 = Date.now();
  let job;
  while (true) {
    await new Promise(res => setTimeout(res, 1500));
    try { job = await api(`/api/ai/jobs/${start.id}`); } catch (e) { continue; }
    const secs = Math.round((Date.now() - t0) / 1000);
    const lines = job.log.map(l => l.msg).join("\n");
    if (job.status === "running") {
      setStatus("", `<strong>${label} with GPT… ${secs}s</strong><div class="log">${esc(lines)}</div>`);
      continue;
    }
    break;
  }
  setBusy(false);
  const summary = `<div class="log">${esc(job.log.map(l => l.msg).join("\n"))}\nTokens: ${job.tokens[0]} in / ${job.tokens[1]} out</div>`;
  if (job.status === "error") { setStatus("bad", `<strong>GPT couldn't finish.</strong>${summary}`); return; }
  const r = job.result;
  if (r.unchanged) { setStatus("ok", `Nothing to fix.${summary}`); return; }
  try { editor.value = pretty(JSON.parse(r.source)) + "\n"; } catch (e) { editor.value = r.source + "\n"; }
  nameInput.value = r.name;
  dirty = false;
  showRenderResult(r, r.name);
  statusBox.insertAdjacentHTML("beforeend", summary);
  await refreshList();
  const meta = diagrams.find(d => d.name === r.name);
  setLinks(r.name, meta && meta.rendered);
}
$("aiCreate").addEventListener("click", () => runAI("create"));
$("aiEdit").addEventListener("click", () => runAI("edit"));
$("aiText").addEventListener("keydown", (e) => {
  if ((e.ctrlKey || e.metaKey) && e.key === "Enter") { e.preventDefault(); runAI("create"); }
});
$("renderBtn").addEventListener("click", render);

// ---------- editor conveniences ----------
editor.addEventListener("keydown", (e) => {
  if ((e.ctrlKey || e.metaKey) && (e.key === "Enter" || e.key === "s")) { e.preventDefault(); render(); return; }
  if (e.key === "Tab" && !e.shiftKey) {
    e.preventDefault();
    const s = editor.selectionStart, t = editor.selectionEnd;
    editor.setRangeText("  ", s, t, "end"); dirty = true;
  }
});
editor.addEventListener("input", () => { dirty = true; });
const updatePos = () => {
  const before = editor.value.slice(0, editor.selectionStart).split("\n");
  $("pos").textContent = `${before.length}:${before[before.length - 1].length + 1}`;
};
["keyup", "click", "select"].forEach(ev => editor.addEventListener(ev, updatePos));
nameInput.addEventListener("blur", () => { nameInput.value = cleanName(nameInput.value); });
nameInput.addEventListener("keydown", (e) => { if (e.key === "Enter") { e.preventDefault(); render(); } });

$("formatBtn").addEventListener("click", () => {
  try { editor.value = pretty(JSON.parse(editor.value)) + "\n"; dirty = true; setStatus("", "Formatted."); }
  catch (e) { setStatus("bad", `Can't format: the JSON isn't valid yet. ${esc(e.message)}`); }
});
$("uploadBtn").addEventListener("click", () => $("file").click());
$("file").addEventListener("change", async (e) => {
  const f = e.target.files[0]; e.target.value = "";
  if (!f || !confirmDiscard()) return;
  editor.value = await f.text();
  nameInput.value = uniqueName(cleanName(f.name)); current = nameInput.value; dirty = true;
  clearPreview(); setLinks(current, false);
  setStatus("", `Loaded ${esc(f.name)}. Press Render to check and save it.`);
});
window.addEventListener("beforeunload", (e) => { if (dirty) { e.preventDefault(); e.returnValue = ""; } });

// ---------- guide ----------
$("guideBtn").addEventListener("click", async () => {
  const q = $("guideQ").value.trim();
  const out = $("guideOut");
  if (!q) { out.textContent = "Describe what you want to show first."; return; }
  out.textContent = "Thinking…";
  const r = await api("/api/guide", { method: "POST", body: JSON.stringify({ q }) });
  if (!r.ok || !r.type) { out.textContent = r.error || "No suggestion for that description."; return; }
  const low = r.confidence === "low"
    ? `<p class="fix">Weak match. Words like state, pipeline, request, or data lineage help.</p>` : "";
  out.innerHTML = `<p><strong>${esc(r.title)}</strong> (${esc(r.type)} diagram)</p>${low}
    <p>${esc(r.summary)}</p>
    ${r.include.length ? `<ul>${r.include.map(i => `<li>${esc(i)}</li>`).join("")}</ul>` : ""}
    <button class="btn light" data-type="${esc(r.type)}">Start a ${esc(r.type)} diagram</button>`;
});
$("guideOut").addEventListener("click", (e) => {
  const b = e.target.closest("button[data-type]"); if (b) newFrom(b.dataset.type);
});

// ---------- start ----------
(async () => {
  const ai = await api("/api/ai/status");
  aiEnabled = ai.enabled;
  $("aiModel").value = ai.model;
  if (ai.enabled) {
    $("aiInfo").textContent = `via ${ai.base}, up to ${ai.maxRounds} rounds of automatic repair`;
  } else {
    $("aiBody").innerHTML = `<p class="ai-off">GPT is off: no OpenAI API key was found. Add <code>OPENAI_API_KEY</code> in Colab Secrets, then rerun the key and server cells.</p>`;
    $("aiPanel").open = false;
  }
  starters = await api("/api/starters");
  await refreshList();
  if (diagrams.length) await openDiagram(diagrams[0].name);
  else { newFrom("architecture"); dirty = false; }
})();
</script>
</body>
</html>

Overwriting /content/studio/index.html


# 5) write small project arch .json

In [5]:
%%writefile /content/studio/starters.json
{
"architecture": {
  "schema_version": 1,
  "diagram_type": "architecture",
  "meta": {"title": "My System Architecture", "visual_preset": "signal-flow"},
  "components": [
    {"id": "users", "type": "external", "label": "Users", "sublabel": "web + mobile", "pos": [40, 250], "size": [120, 60]},
    {"id": "web", "type": "frontend", "label": "Web App", "sublabel": "React", "pos": [220, 250], "size": [130, 60]},
    {"id": "api", "type": "backend", "label": "API Server", "sublabel": "REST", "pos": [430, 250], "size": [130, 60]},
    {"id": "cache", "type": "database", "label": "Cache", "sublabel": "Redis", "pos": [430, 100], "size": [130, 60]},
    {"id": "db", "type": "database", "label": "Database", "sublabel": "PostgreSQL", "pos": [640, 250], "size": [130, 60]},
    {"id": "queue", "type": "messagebus", "label": "Job Queue", "sublabel": "async work", "pos": [430, 400], "size": [130, 60]},
    {"id": "worker", "type": "backend", "label": "Worker", "sublabel": "background jobs", "pos": [640, 400], "size": [130, 60]}
  ],
  "boundaries": [
    {"kind": "region", "label": "Cloud", "wraps": ["web", "api", "cache", "db", "queue", "worker"]}
  ],
  "connections": [
    {"id": "u-web", "from": "users", "to": "web", "label": "HTTPS", "variant": "emphasis"},
    {"id": "web-api", "from": "web", "to": "api", "label": "JSON"},
    {"id": "api-cache", "from": "api", "to": "cache", "label": "read", "fromSide": "top", "toSide": "bottom", "labelDy": -66},
    {"id": "api-db", "from": "api", "to": "db", "label": "SQL"},
    {"id": "api-q", "from": "api", "to": "queue", "label": "enqueue", "variant": "dashed", "fromSide": "bottom", "toSide": "top", "labelDy": 66},
    {"id": "q-w", "from": "queue", "to": "worker"}
  ]
},
"sequence": {
  "schema_version": 1,
  "diagram_type": "sequence",
  "meta": {"title": "My Request Sequence", "viewBox": [960, 500], "animation": "trace"},
  "participants": [
    {"id": "client", "type": "frontend", "label": "Client", "sublabel": "browser"},
    {"id": "api", "type": "backend", "label": "API", "sublabel": "handler"},
    {"id": "cache", "type": "database", "label": "Cache", "sublabel": "Redis"},
    {"id": "db", "type": "database", "label": "Database", "sublabel": "PostgreSQL"}
  ],
  "messages": [
    {"id": "m1", "from": "client", "to": "api", "y": 160, "label": "GET /items", "variant": "emphasis"},
    {"id": "m2", "from": "api", "to": "cache", "y": 195, "label": "read cache"},
    {"id": "m3", "from": "cache", "to": "api", "y": 225, "label": "miss", "variant": "return"},
    {"id": "m4", "from": "api", "to": "db", "y": 260, "label": "SELECT items", "variant": "emphasis"},
    {"id": "m5", "from": "db", "to": "api", "y": 290, "label": "rows", "variant": "return"},
    {"id": "m6", "from": "api", "to": "cache", "y": 320, "label": "set cache", "variant": "dashed"},
    {"id": "m7", "from": "api", "to": "client", "y": 355, "label": "200 JSON", "variant": "return"}
  ]
},
"workflow": {
  "schema_version": 1,
  "diagram_type": "workflow",
  "meta": {"title": "My Workflow", "animation": "trace"},
  "lanes": [
    {"id": "dev", "label": "Developer"},
    {"id": "ci", "label": "CI"},
    {"id": "ops", "label": "Production"}
  ],
  "mainPath": ["commit", "build", "test", "deploy"],
  "nodes": [
    {"id": "commit", "lane": "dev", "col": 0, "type": "frontend", "label": "Commit", "sublabel": "push change"},
    {"id": "build", "lane": "ci", "col": 1, "type": "backend", "label": "Build", "sublabel": "compile"},
    {"id": "test", "lane": "ci", "col": 3, "type": "security", "label": "Test", "sublabel": "quality gate"},
    {"id": "deploy", "lane": "ops", "col": 4, "type": "cloud", "label": "Deploy", "sublabel": "release"}
  ],
  "edges": [
    {"from": "commit", "to": "build"},
    {"from": "build", "to": "test"},
    {"from": "test", "to": "deploy", "variant": "emphasis"}
  ]
},
"dataflow": {
  "schema_version": 1,
  "diagram_type": "dataflow",
  "meta": {"title": "My Data Flow", "animation": "trace"},
  "stages": [
    {"label": "Sources"},
    {"label": "Processing"},
    {"label": "Storage"}
  ],
  "nodes": [
    {"id": "app", "type": "frontend", "label": "App", "sublabel": "events", "stage": 0, "row": 0},
    {"id": "etl", "type": "backend", "label": "ETL Job", "sublabel": "clean + join", "stage": 1, "row": 0},
    {"id": "wh", "type": "database", "label": "Warehouse", "sublabel": "analytics", "stage": 2, "row": 0}
  ],
  "flows": [
    {"from": "app", "to": "etl", "label": "raw events", "variant": "emphasis"},
    {"from": "etl", "to": "wh", "label": "clean rows"}
  ]
},
"lifecycle": {
  "schema_version": 1,
  "diagram_type": "lifecycle",
  "meta": {"title": "My Order Lifecycle", "animation": "trace"},
  "lanes": [
    {"id": "main", "label": "Main states"},
    {"id": "exceptions", "label": "Exceptions"}
  ],
  "states": [
    {"id": "created", "type": "start", "label": "Created", "sublabel": "order placed", "lane": "main", "col": 0},
    {"id": "paid", "type": "active", "label": "Paid", "sublabel": "payment ok", "lane": "main", "col": 1},
    {"id": "shipped", "type": "active", "label": "Shipped", "sublabel": "in transit", "lane": "main", "col": 2},
    {"id": "delivered", "type": "success", "label": "Delivered", "sublabel": "done", "lane": "main", "col": 3},
    {"id": "cancelled", "type": "failure", "label": "Cancelled", "sublabel": "user stopped", "lane": "exceptions", "col": 1}
  ],
  "transitions": [
    {"id": "t-cancel", "from": "paid", "to": "cancelled", "variant": "security", "route": "drop"}
  ]
}
}

Overwriting /content/studio/starters.json


# 6) integrate OpenAI and a private access key for the studio link

In [6]:
import os, secrets

OPENAI_MODEL = "gpt-5.6-terra"   # also: gpt-5.6-luna (cheapest), gpt-5.6-sol, gpt-6-astra (most capable)

OPENAI_API_KEY = ""
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY") or ""
except Exception:
    pass
OPENAI_API_KEY = OPENAI_API_KEY or os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    try:
        from getpass import getpass
        OPENAI_API_KEY = getpass("OpenAI API key (press Enter to skip GPT): ").strip()
    except Exception:
        pass

STUDIO_KEY = globals().get("STUDIO_KEY") or secrets.token_urlsafe(16)   # stays the same when rerun
print("GPT:", f"on ({OPENAI_MODEL})" if OPENAI_API_KEY else "off (no API key)")

GPT: on (gpt-5.6-terra)


# 7) Start the studio server on port 8000


In [7]:
import http.client, sys, time

PORT = 8000
!pkill -f "[/]content/studio/server.py" || true
time.sleep(1)

env = {**os.environ, "PORT": str(PORT), "ARCHIFY_CLI": "/content/archify/archify/bin/archify.mjs",
       "STUDIO_KEY": STUDIO_KEY, "OPENAI_API_KEY": OPENAI_API_KEY, "OPENAI_MODEL": OPENAI_MODEL}
# Other OpenAI-compatible providers: uncomment and edit
# env["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"; env["OPENAI_API_STYLE"] = "chat"
studio = subprocess.Popen([sys.executable, "/content/studio/server.py"], cwd="/content/studio", env=env,
                          stdout=open("/content/studio/server.log", "w"), stderr=subprocess.STDOUT)

def studio_answers():
    """True only if this studio (with this access key) answers on the port."""
    try:
        c = http.client.HTTPConnection("127.0.0.1", PORT, timeout=2)
        c.request("GET", f"/?key={STUDIO_KEY}")
        return c.getresponse().status == 302
    except Exception:
        return False

for _ in range(30):
    if studio.poll() is not None:
        print("Studio stopped (another program may be using port 8000):\n" + open("/content/studio/server.log").read())
        break
    if studio_answers():
        print(f"Studio is running on port {PORT}. GPT: {'on' if OPENAI_API_KEY else 'off'}")
        break
    time.sleep(0.5)
else:
    print("Studio didn't answer. Check /content/studio/server.log")

Studio is running on port 8000. GPT: on


# 8) Open the Cloudflare quick tunnel


In [8]:
import re
from IPython.display import HTML, display

if globals().get("tunnel") and tunnel.poll() is None and globals().get("PUBLIC_URL"):
    print("Tunnel already running")
else:
    !pkill -f "[c]loudflared tunnel" || true
    PUBLIC_URL = None
    tunnel = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2",
         "--url", f"http://127.0.0.1:{PORT}"],
        stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        time.sleep(1)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
        if m:
            PUBLIC_URL = m.group(0)
            break
        if tunnel.poll() is not None:
            break

if PUBLIC_URL:
    link = f"{PUBLIC_URL}/?key={STUDIO_KEY}"
    display(HTML(f'<p style="font-size:18px">Open Archify Studio: <a href="{link}" target="_blank">{link}</a></p>'
                 '<p>This link includes your private access key; share it only with people who may use your OpenAI credits.<br>'
                 'If the page doesn\'t load right away, wait about 15 seconds and refresh.</p>'))
else:
    print("No URL yet. Last lines of tunnel.log:")
    print("".join(open("/content/tunnel.log").readlines()[-15:]))

# 9) check that the public link responds


In [9]:
import requests

if not globals().get("PUBLIC_URL"):
    print("No public URL yet. Run cell 8 first.")
else:
    for attempt in range(12):
        try:
            s = requests.Session()
            s.get(f"{PUBLIC_URL}/?key={STUDIO_KEY}", timeout=10)
            r = s.get(f"{PUBLIC_URL}/api/diagrams", timeout=10)
            print("Status:", r.status_code, "| saved diagrams:", len(r.json()))
            break
        except Exception:
            print("Not reachable yet, retrying...")
            time.sleep(5)
print("".join(open("/content/studio/server.log").readlines()[-10:]))

Not reachable yet, retrying...
Not reachable yet, retrying...
Not reachable yet, retrying...
Not reachable yet, retrying...
Not reachable yet, retrying...
Not reachable yet, retrying...
Status: 200 | saved diagrams: 6
Archify Studio on http://127.0.0.1:8000  (data: /content/studio/data, GPT: on)
[15:16:51] GET / -> 302
[15:17:26] GET / -> 302
[15:17:26] GET / -> 200
[15:17:26] GET /api/diagrams -> 200



In [ ]:
import time

while True:
    time.sleep(60)

# 10) Stop the tunnel and the studio

In [10]:
# !pkill -f "[c]loudflared tunnel" || true
# !pkill -f "[/]content/studio/server.py" || true
# PUBLIC_URL = None
# print("Stopped")